In [1]:
import wandb
import pandas as pd
import numpy as np

import json

api = wandb.Api()

In [2]:
def linearize_dict(d, parent_key='', sep='.'):
    """
    Linearizes a nested dictionary into a flat dictionary with keys in the format `key1.key2`.

    :param d: The dictionary to linearize.
    :param parent_key: The current key being processed (used for recursion).
    :param sep: The separator between keys.
    :return: A flattened dictionary.
    """
    items = []
    
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.extend(linearize_dict(v, new_key, sep).items())
        else:
            items.append((new_key, v))
    
    return dict(items)

def get_runs_df(runs):
    runs_list = []
    for run in runs: 
        runs_list.append({
            **run.summary._json_dict,
            **linearize_dict({k: v for k,v in run.config.items()
            if not k.startswith('_')}),
            **linearize_dict(run.metadata if run.metadata else {}),
            **{"name": run.name, "id": run.id}
            })

    runs_df = pd.DataFrame(runs_list)
    return runs_df

In [3]:
iauc_dauc_runs = api.runs(
    "pasqualedem/affex",
    filters={
        "$and": [
            {"config.metric.n_steps": 75},
            {"tags": {"$nin": ["Broken"]}},
            {"$or": [
                {'config.dataset.datasets.val_deepglobe_N1K5.name': "deepglobe"},
                {'config.dataset.datasets.val_lung_N1K5.name': "lung"},
                {'config.dataset.datasets.val_isic_N1K5.name': "isic"},
            ]}
        ]
    }
)

In [4]:
iauc_dauc_runs_df = get_runs_df(iauc_dauc_runs)

In [26]:
res_df = iauc_dauc_runs_df.copy()

dataset_names = ["val_lung_N1K5", "val_isic_N1K5", "val_deepglobe_N1K5"]

dataset_cols = [f'dataset.datasets.{name}.name' for name in dataset_names]
res_df["Dataset"] = res_df[dataset_cols].fillna("").sum(axis=1)

iauc_cols = [f"{name}_iauc" for name in dataset_names]
dauc_cols = [f"{name}_dauc" for name in dataset_names]
assert res_df[iauc_cols].isna().any(axis=1).all(), "More than one dataset is present in the same run, please check the dataset columns."
assert res_df[dauc_cols].isna().any(axis=1).all(), "More than one dataset is present in the same run, please check the dataset columns."


renamings_dict = {
    "explainer.name": "Explanation Method",
    "config.metric.n_steps": "N. Steps",
    "config.metric.iauc": "IAUC",
    "config.metric.dauc": "DAUC",
    "model": "Model",
}
res_df = res_df.rename(columns=renamings_dict)

value_renamings_dict = {
    "integrated_gradients": "Integrated Gradients",
    "saliency": "Saliency",
    "affinity": "Affinity Explainer (ours)",
    "random": "Random",
    "pascal": "Pascal 5^i",
    "coco": "COCO 20^i",
    "dcama": "DCAMA",
    "dmtnet": "DMTNet",
    "isic": "ISIC",
    "lung": "Lung",
    "deepglobe": "Deepglobe",
}
res_df["Explanation Method"] = res_df["Explanation Method"].replace(value_renamings_dict)
res_df["Dataset"] = res_df["Dataset"].replace(value_renamings_dict)
res_df["Model"] = res_df["Model"].replace(value_renamings_dict)


res_df["IAUC"] = res_df[iauc_cols].fillna(0).sum(axis=1)
res_df["DAUC"] = res_df[dauc_cols].fillna(0).sum(axis=1)

res_df["Diff."] = res_df["IAUC"] - res_df["DAUC"]

cols = ["Model", "Explanation Method", "Dataset"]
res_df = res_df[cols + ["IAUC", "DAUC", "Diff."]]

# Define custom sort order
dataset_order = {"Deepglobe": 0, "ISIC": 1, "Lung": 2}
metric_order = {'IAUC': 0, 'DAUC': 1, 'Diff.': 2}
method_order = {
    'Random': 0,
    'Integrated Gradients': 1,
    'Saliency': 2,
    'Affinity Explainer': 3,
}
model_order = {
    'DCAMA': 0,
    'DMTNet': 1,
}

print(res_df)

res_df = res_df.pivot_table(
    index=['Dataset', 'Explanation Method'],
    # columns=,
    values=['IAUC', 'DAUC', 'Diff.']
).sort_index(axis=1)


new_cols = ['IAUC', 'DAUC', 'Diff.']
res_df = res_df[new_cols]
# Sort by Model and then by Explanation Method using the provided orderings
res_df = res_df.sort_values(
    by=['Dataset', 'Explanation Method'],  # this ensures level names exist and are used
    key=lambda x: x.map(dataset_order) if x.name == 'Dataset' else x.map(method_order)
)

iauc_dauc_df = (res_df*100).round(2)

iauc_dauc_df

     Model         Explanation Method    Dataset      IAUC      DAUC     Diff.
0   DMTNet                   Saliency       ISIC  0.732147  0.781361 -0.049214
1   DMTNet       Integrated Gradients       ISIC  0.653066  0.730476 -0.077410
2   DMTNet  Affinity Explainer (ours)       ISIC  0.790059  0.644435  0.145624
3   DMTNet                     Random       ISIC  0.739172  0.739680 -0.000508
4   DMTNet                     Random       ISIC  0.739172  0.739680 -0.000508
5   DMTNet                   Saliency       Lung  0.812691  0.746967  0.065724
6   DMTNet       Integrated Gradients       Lung  0.719256  0.748969 -0.029713
7   DMTNet  Affinity Explainer (ours)       Lung  0.902064  0.643075  0.258989
8   DMTNet                   Saliency  Deepglobe  0.461447  0.558001 -0.096554
9   DMTNet                     Random       Lung  0.732055  0.732375 -0.000320
10  DMTNet       Integrated Gradients  Deepglobe  0.245286  0.375693 -0.130407
11  DMTNet  Affinity Explainer (ours)  Deepglobe  0.

IAUC   DAUC  Diff.
Dataset   Explanation Method                            
Deepglobe Random                     30.92  30.98  -0.05
          Integrated Gradients       24.53  37.57 -13.04
          Saliency                   46.14  55.80  -9.66
          Affinity Explainer (ours)  53.12  38.63  14.49
ISIC      Random                     73.92  73.97  -0.05
          Integrated Gradients       65.31  73.05  -7.74
          Saliency                   73.21  78.14  -4.92
          Affinity Explainer (ours)  79.01  64.44  14.56
Lung      Random                     73.21  73.24  -0.03
          Integrated Gradients       71.93  74.90  -2.97
          Saliency                   81.27  74.70   6.57
          Affinity Explainer (ours)  90.21  64.31  25.90

In [11]:
from tabulate import tabulate

latex_tab = tabulate(res_df, headers='keys', tablefmt='latex_booktabs', showindex=True)
print(latex_tab)

\begin{tabular}{lrrrrrrrrr}
\toprule
                                         &   ('Deepglobe', 'IAUC') &   ('Deepglobe', 'DAUC') &   ('Deepglobe', 'Diff.') &   ('ISIC', 'IAUC') &   ('ISIC', 'DAUC') &   ('ISIC', 'Diff.') &   ('Lung', 'IAUC') &   ('Lung', 'DAUC') &   ('Lung', 'Diff.') \\
\midrule
 ('DMTNet', 'Random')                    &                0.309229 &                0.309751 &               -0.0005225 &           0.739172 &           0.73968  &        -0.000508215 &           0.732055 &           0.732375 &        -0.000320113 \\
 ('DMTNet', 'Integrated Gradients')      &                0.245286 &                0.375693 &               -0.130407  &           0.653066 &           0.730476 &        -0.0774099   &           0.719256 &           0.748969 &        -0.0297134   \\
 ('DMTNet', 'Saliency')                  &                0.461447 &                0.558001 &               -0.0965538 &           0.732147 &           0.781361 &        -0.0492141   &           0.812

## IAUC %

In [12]:
iauc_p_runs = api.runs(
    "pasqualedem/affex",
    filters={
        "$and": [
            {"config.metric.percentage": {"$ne": None}},
            {"tags": {"$nin": ["Broken"]}},
            {"$or": [
                {'config.dataset.datasets.val_deepglobe_N1K5.name': "deepglobe"},
                {'config.dataset.datasets.val_lung_N1K5.name': "lung"},
                {'config.dataset.datasets.val_isic_N1K5.name': "isic"},
            ]}
        ]
    }
)

In [13]:
iauc_p_runs_df = get_runs_df(iauc_p_runs)

In [30]:
res_df = iauc_p_runs_df.copy()

dataset_names = ["val_lung_N1K5", "val_isic_N1K5", "val_deepglobe_N1K5"]

dataset_cols = [f'dataset.datasets.{name}.name' for name in dataset_names]
res_df["Dataset"] = res_df[dataset_cols].fillna("").sum(axis=1)

iauc_cols = [f"{name}_iauc" for name in dataset_names]
assert res_df[iauc_cols].isna().any(axis=1).all(), "More than one dataset is present in the same run, please check the dataset columns."


renamings_dict = {
    "explainer.name": "Explanation Method",
    "config.metric.n_steps": "N. Steps",
    "config.metric.iauc": "IAUC",
    "config.metric.dauc": "DAUC",
    "model": "Model",
    "metric.percentage": "Percentage",
    "metric.loss": "Loss",
}
res_df = res_df.rename(columns=renamings_dict)

value_renamings_dict = {
    "integrated_gradients": "Integrated Gradients",
    "saliency": "Saliency",
    "affinity": "Affinity Explainer (ours)",
    "random": "Random",
    "pascal": "Pascal 5^i",
    "coco": "COCO 20^i",
    "dcama": "DCAMA",
    "dmtnet": "DMTNet",
    "isic": "ISIC",
    "lung": "Lung",
    "deepglobe": "Deepglobe",
}
res_df["Explanation Method"] = res_df["Explanation Method"].replace(value_renamings_dict)
res_df["Dataset"] = res_df["Dataset"].replace(value_renamings_dict)
res_df["Model"] = res_df["Model"].replace(value_renamings_dict)


res_df["iAUC"] = res_df[iauc_cols].fillna(0).sum(axis=1)
res_df["Loss"] = res_df["Loss"].fillna(False)

cols = ["Model", "Explanation Method", "Dataset", "Percentage", "Loss"]
res_df = res_df[cols + ["iAUC"]]

# Define custom sort order
dataset_order = {"Deepglobe": 0, "ISIC": 1, "Lung": 2}
method_order = {
    'Random': 0,
    'Integrated Gradients': 1,
    'Saliency': 2,
    'Affinity Explainer': 3,
}
model_order = {
    'DCAMA': 0,
    'DMTNet': 1,
}

res_df = res_df.pivot_table(
    index=['Dataset', 'Explanation Method'],
    columns=['Percentage', "Loss"],
    values='iAUC'
).sort_index(axis=1)

perc_map = {
    False: "IAUC@",
    True: "mIoULoss@"
}

# Flatten the MultiIndex columns
# Collapse last two levels into one
res_df.columns = pd.Index([
    f"{perc_map[lvl1]}{lvl0}" for lvl0, lvl1 in res_df.columns
])


# Sort by Model and then by Explanation Method using the provided orderings
res_df = res_df.sort_values(
    by=['Dataset', 'Explanation Method'],  # this ensures level names exist and are used
    key=lambda x: x.map(dataset_order) if x.name == 'Dataset' else x.map(method_order)
)

iauc_p_df = (res_df*100).round(2)

iauc_p_df

/tmp/ipykernel_3860822/3154122205.py:42: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  res_df["Loss"] = res_df["Loss"].fillna(False)


IAUC@0.01  mIoULoss@0.01  mIoULoss@0.05
Dataset   Explanation Method                                                
Deepglobe Random                         42.85          14.89          27.60
          Integrated Gradients           42.22          15.96          25.01
          Saliency                       44.21          13.50          14.40
          Affinity Explainer (ours)      43.52          12.94          13.10
ISIC      Random                         70.72          22.49          18.71
          Integrated Gradients           75.36          20.66          15.77
          Saliency                       79.63          19.06          14.13
          Affinity Explainer (ours)      78.20          18.08          12.47
Lung      Random                         62.89          27.76          56.42
          Integrated Gradients           71.24          25.91          22.00
          Saliency                       72.42          25.96          24.89
          Affinity Explainer (ours)      79.59          27.01          26.52

---

In [32]:
# joining iauc_dauc_df and iauc_p_df on index
final = iauc_dauc_df.join(iauc_p_df)

metric_order = {
    'IAUC': 0,
    'DAUC': 1,
    'Diff.': 2,
    'IAUC@0.01': 3,
    'IAUC@0.05': 4,
    'mIoULoss@0.01': 5,
    'mIoULoss@0.05': 6,
}

# new_cols  = sorted(final.columns, key=lambda x: (dataset_order[x[0]], metric_order[x[1]]))
# final = final[new_cols]
final

IAUC   DAUC  Diff.  IAUC@0.01  \
Dataset   Explanation Method                                          
Deepglobe Random                     30.92  30.98  -0.05      42.85   
          Integrated Gradients       24.53  37.57 -13.04      42.22   
          Saliency                   46.14  55.80  -9.66      44.21   
          Affinity Explainer (ours)  53.12  38.63  14.49      43.52   
ISIC      Random                     73.92  73.97  -0.05      70.72   
          Integrated Gradients       65.31  73.05  -7.74      75.36   
          Saliency                   73.21  78.14  -4.92      79.63   
          Affinity Explainer (ours)  79.01  64.44  14.56      78.20   
Lung      Random                     73.21  73.24  -0.03      62.89   
          Integrated Gradients       71.93  74.90  -2.97      71.24   
          Saliency                   81.27  74.70   6.57      72.42   
          Affinity Explainer (ours)  90.21  64.31  25.90      79.59   

                                     mIoULoss@0.01  mIoULoss@0.05  
Dataset   Explanation Method                                       
Deepglobe Random                             14.89          27.60  
          Integrated Gradients               15.96          25.01  
          Saliency                           13.50          14.40  
          Affinity Explainer (ours)          12.94          13.10  
ISIC      Random                             22.49          18.71  
          Integrated Gradients               20.66          15.77  
          Saliency                           19.06          14.13  
          Affinity Explainer (ours)          18.08          12.47  
Lung      Random                             27.76          56.42  
          Integrated Gradients               25.91          22.00  
          Saliency                           25.96          24.89  
          Affinity Explainer (ours)          27.01          26.52

In [33]:
# turn the two level index into two columns
final_to_latex = final.reset_index()

latex_tab = tabulate(final_to_latex, headers='keys', tablefmt='latex_booktabs', showindex=False)
print(latex_tab)

\begin{tabular}{llrrrrrr}
\toprule
 Dataset   & Explanation Method        &   IAUC &   DAUC &   Diff. &   IAUC@0.01 &   mIoULoss@0.01 &   mIoULoss@0.05 \\
\midrule
 Deepglobe & Random                    &  30.92 &  30.98 &   -0.05 &       42.85 &           14.89 &           27.6  \\
 Deepglobe & Integrated Gradients      &  24.53 &  37.57 &  -13.04 &       42.22 &           15.96 &           25.01 \\
 Deepglobe & Saliency                  &  46.14 &  55.8  &   -9.66 &       44.21 &           13.5  &           14.4  \\
 Deepglobe & Affinity Explainer (ours) &  53.12 &  38.63 &   14.49 &       43.52 &           12.94 &           13.1  \\
 ISIC      & Random                    &  73.92 &  73.97 &   -0.05 &       70.72 &           22.49 &           18.71 \\
 ISIC      & Integrated Gradients      &  65.31 &  73.05 &   -7.74 &       75.36 &           20.66 &           15.77 \\
 ISIC      & Saliency                  &  73.21 &  78.14 &   -4.92 &       79.63 &           19.06 &           14.13